# C01 — 캐리어 28번 에어 누수 (2026-10-17, X-029)

정리본: [`cases/C01.md`](cases/C01.md). 이 노트북은 그 근거를 다시 만든다 (DB 는 캐시가 있으면 안 감).

| 단계 | 내용 |
|---|---|
| 1 | 일지 내용 |
| 2 | 가동일 기록(하루 단위)을 운영 판정 함수에 넣기 — 그날 모델이 내렸을 판정 |
| 3 | DB → 운영 코어 재생 (10-16 ~ 10-27) — 통과 하나하나 |
| 4 | 모델 화면에 떴을 내용 (재생의 판정 결과) |
| 5 | 실제 엣지 경고 (10-26 W0073) |
| 6 | 유닛 사이클을 대차별로 — 유닛 지표가 이 대차를 보나 |
| 7 | **딥 리서치 — 에어 누수가 나면 주입 시간이 길어지나** (PLC 로직 · 에어 관련 신호 · 2라인 누수 6건) |

In [ ]:
import sys, json; sys.path.insert(0, "..")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from review_lib import paths, db, replay, backend, plot
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
FIGS = paths.CHECK_DIRS["conveyor"] / "figs"; FIGS.mkdir(exist_ok=True)
WAGON, SENSORS = 28, ["P00451", "P00459"]          # 전진·후진 Unit 캐리어 진입감지
ACT = (pd.Timestamp("2026-10-17 09:30"), pd.Timestamp("2026-10-17 10:30"))

## 1. 일지

In [ ]:
acts = pd.read_csv(paths.ACTIONS_DIR / "actions_conveyor.csv")
acts[acts.incident_id == "C01"].T

## 2. 가동일 기록 → 운영 판정 (하루 단위)

`v2_2_1/backend/data/conveyor_core_history.json` — 09-10 ~ 10-19 씨앗(배치), 10-26 ~ 11-05 실시간 누적. **10-20 ~ 10-25 는 비어 있다.**

In [ ]:
backend.use("v2.2")
from tools_yeji.conveyor_models.core_judge import judge, sensor_level
from tools_yeji.conveyor_models.core_params import LONG_REF
H = pd.DataFrame(json.load(open(paths.MODEL_VERSIONS["v2.2"]["backend"] / "data" / "conveyor_core_history.json")))
H["day"] = pd.to_datetime(H.day)
J = judge(H)
w = J[(J.wagon == WAGON) & J.sensor.isin(SENSORS) & (J.day >= "2026-09-08")]
w.pivot_table(index="day", columns="sensor", values=["state", "rearm", "width_ex", "rise_off"], aggfunc="first").round(2)

In [ ]:
# 같은 날 후진 진입(P00459)에서 이상인 대차 — 10-17·10-19 은 28번 하나뿐
x = J[(J.sensor == "P00459") & (J.day.between("2026-10-10", "2026-10-31")) & (J.state != "정상")]
x.groupby("day").apply(lambda g: ", ".join(f"{a}:{b}" for a, b in zip(g.wagon, g.state)), include_groups=False)

## 3. DB → 운영 코어 재생 (통과 하나하나)

기록은 10-16 이전 날만 넘긴다 → 10-16 부터 모델이 돌았다면 무엇을 봤을지.
재생 결과는 가동일 기록과 맞는다 (10-16·17 검증: 앞 끝 위치 동일, 재감지 차이 ≤ 0.011).

In [ ]:
TAGS = db.model_tags("conveyor", "v2.2")
raw = db.fetch_range("2026-10-16", "2026-10-27", TAGS, group="conveyor")
hist = json.load(open(paths.MODEL_VERSIONS["v2.2"]["backend"] / "data" / "conveyor_core_history.json"))
P, E, core = replay.conveyor_core(raw, history=[r for r in hist if r["day"] < "2026-10-16"],
                                  eval_every="15min", units=("md_fwd", "md_rev"))
P["day"] = (P.t - pd.Timedelta(hours=6)).dt.normalize()
D = P.groupby(["day", "sensor"]).dist.median().rename("D")        # 그날 센서의 대차 공통 위치
P = P.join(D, on=["day", "sensor"]); P["front"] = P.dist - P.D      # 앞 끝 편차 (−: 표식보다 앞에서 켜짐)
p28 = P[(P.wagon == WAGON) & P.sensor.isin(SENSORS)].sort_values("t")
p28.groupby([p28.t.dt.date, "sensor"]).agg(n=("t", "size"), rearm=("rearm", "mean"),
                                          front=("front", "median"), width=("width", "median")).unstack().round(2)

In [ ]:
# 바뀐 순간: 10-16 마지막 통과 vs 10-17 첫 통과, 10-22 마지막 재감지 vs 이후
r = p28[p28.sensor == "P00459"]
print("10-16 마지막 통과", r[r.t < "2026-10-17 06:00"].tail(2)[["t", "rearm", "front"]].to_string())
print("10-17 첫 통과   ", r[r.t > "2026-10-17 05:00"].head(3)[["t", "rearm", "front"]].to_string())
last_dirty = r[(r.t < "2026-10-23") & (r.rearm > 0)].t.max()
print("마지막 재감지 통과", last_dirty, "→ 그 뒤 첫 통과", r[r.t > last_dirty].t.min())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
for ax, s in zip(axes, SENSORS):
    q = p28[p28.sensor == s]
    others = P[(P.sensor == s) & (P.wagon != WAGON)].set_index("t").rearm.resample("1h").mean()
    ax.plot(others.index, others.values, color="gray", lw=0.8, label="다른 대차 평균 (1시간)")
    ax.scatter(q.t, q.rearm, s=8, color="tab:red", label=f"대차 {WAGON} 통과별 재감지 수")
    ax.axvspan(*ACT, color="tab:blue", alpha=0.3, label="일지 조치 10-17 09:30~10:30")
    ax.axvspan(pd.Timestamp("2026-10-22 09:56"), pd.Timestamp("2026-10-22 11:34"), color="tab:green", alpha=0.3,
               label="재감지 사라짐 10-22 (일지 없음)")
    ax.axvline(pd.Timestamp("2026-10-26 07:09"), color="k", ls=":", label="엣지 경고 W0073 10-26 07:09")
    ax.set_ylabel(s); ax.set_ylim(-0.2, 4)
axes[0].legend(loc="upper right", fontsize=8, ncol=3)
axes[0].set_title(f"대차 {WAGON} — 전진(P00451)·후진(P00459) Unit 캐리어 진입감지 재감지")
fig.tight_layout(); fig.savefig(FIGS / "C01_wagon28_rearm.png", dpi=110, bbox_inches="tight")

In [ ]:
# 10-16 ~ 10-17 확대: 앞 끝 편차 — 매 통과 같은 자리(후진 −6, 전진 −14)에서 먼저 켜짐 = 대차에 붙은 고정된 무언가
fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)
for ax, s in zip(axes, SENSORS):
    q = p28[(p28.sensor == s) & p28.t.between("2026-10-16 06:00", "2026-10-18 01:00")]
    ax.scatter(q.t, q.front, s=10, c=np.where(q.rearm > 0, "tab:red", "tab:gray"))
    ax.axvspan(*ACT, color="tab:blue", alpha=0.3)
    ax.set_ylabel(f"{s}\n앞 끝 편차"); ax.axhline(0, color="k", lw=0.5)
axes[0].set_title("대차 28 앞 끝 편차 (빨강 = 그 통과에서 재감지)")
fig.tight_layout(); fig.savefig(FIGS / "C01_wagon28_front.png", dpi=110, bbox_inches="tight")

## 4. 모델 화면에 떴을 내용 (재생 판정, 15분마다)

- P00451(전진)은 `센서 재감지`(대차 20대 넘게 재감지) 플래그가 늘 켜져 있어 **대차 경고가 전부 센서 경고 하나로 묶인다** → 28번 변화는 따로 안 보인다.
- P00459(후진)는 28번만 `이물(분리)` → 점검 추천(등급 2) 경고 `wagon_rearm`.

In [ ]:
w28 = E[E.wagon == WAGON]
print(w28.groupby(["sensor", "state"]).t.agg(["min", "max", "count"]))
fl = E[E.wagon.isna() & (E["flags"].fillna("") != "")]
print(fl.groupby(["sensor", "flags"]).t.agg(["min", "max", "count"]))

## 5. 실제 엣지 경고 — W0073 (10-26 07:09 ~ 11:43)

`since: "10-17"` — 엣지 가동일 기록이 10-19 씨앗에서 멈춰 있어(10-20~25 기록 없음) 10-17·10-19 의 이물 상태를 이어받았다.
같은 조건(씨앗 기록 + 10-26 02:15 시작)으로 재생해도 이 경고는 **재현되지 않는다** — 그날 28번 통과는 전부 깨끗했다. 엣지 내부 상태 확인이 필요하다.

In [ ]:
ev = pd.read_csv(paths.EPISODES_DIR / "warning_events.csv", low_memory=False, parse_dates=["ts"])
ev[(ev.key == "P00459:wagons") & ev.ts.between("2026-10-26", "2026-10-27")][["ts", "event", "active", "message", "since"]].head(12)

## 6. 유닛 사이클을 대차별로 — 유닛 지표가 28번을 보나

운영 유닛 모델은 하루 평균만 쓴다. 사이클마다 직전 진입감지 통과의 대차를 붙여(지연 2.5–3.6초로 한 덩어리) 28번과 나머지를 나눈다.

In [ ]:
C = core.unit_cycles
cols = ["stop_fwd_time", "stop_rev_time", "catch_cnt", "drag_cnt", "air_time", "return_time", "nozzle_rev_time"]
out = []
for u in ["md_fwd", "md_rev"]:
    a = replay.attach_wagon(C, P, u)
    a["day"] = (a.t - pd.Timedelta(hours=6)).dt.date
    a["grp"] = np.where(a.wagon == WAGON, "28", "나머지")
    m = a[a.day.astype(str).isin(["2026-10-16", "2026-10-17", "2026-10-21", "2026-10-23"])].groupby(["day", "grp"])[cols].mean()
    out.append(m.assign(unit=u))
pd.concat(out).round(3)

## 7. 딥 리서치 — 에어 누수를 데이터로 확인할 수 있나

**질문**: 캐리어에 에어 누수가 있으면 에어 주입 시간이 길어지나? 그걸 데이터로 확인할 수 있나?

| 단계 | 확인 |
|---|---|
| 7-1 | PLC 로직 — 세 유닛이 하는 일과 주입 시간을 정하는 것 |
| 7-2 | 캐리어 에어 상태를 알려 줄 PLC 신호가 있나 (수집·생존 여부) |
| 7-3 | 에어백 유닛 사이클을 대차에 붙이기 |
| 7-4 | 컨베이어 속도가 대차마다 다르다 — 주입 시간 비교 전에 걸러야 할 것 |
| 7-5 | 속도를 뺀 대차별 주입 시간 — C01(대차 28)·C02(대차 18) |
| 7-6 | 2라인 캐리어 누수 기록 6건 전부 |
| 7-7 | 결론 |

### 7-1. PLC 로직 — 주입 시간은 무엇으로 정해지나

`HDI_Relay/backend/IL_file/Air_Supply_Unit.il` (UTF-16). 세 유닛의 로직은 같은 틀이다.

| 유닛 | IL 이름 | 하는 일 | 카운터 · 리셋 센서 | 주입 최대 |
|---|---|---|---|---|
| 에어백 (`airbag`) | Air Supply Unit #1 **Air Bag용** | 캐리어 에어백에 공기 주입 | R00050 · **P00154 클로징 진입** | 3초 (`Open_Time 30`) |
| 전진 (`md_fwd`) | Air Supply Unit #2 **중형CY 전진용** | 중형 실린더 전진 | R00055 · P00451 | 10초 (`100`) |
| 후진 (`md_rev`) | Air Supply Unit #3 **중형CY 후진용** | 중형 실린더 후진 | R00060 · P00459 | 10초 |

주입 SOL은 노즐 후진감지 OFF 뒤 0.5초에 켜진다. **이동대가 끝단에 닿거나(Rev_PX) 최대 시간이 지나면 꺼진다. 압력·유량 조건은 어디에도 없다.**
→ 원리상 주입 시간은 "이동대가 대차에 끌려간 시간"이다. 캐리어가 새도 길어지지 않는다.

In [ ]:
from pathlib import Path
IL = Path("/Users/jeonjae/Desktop/KTA/HDI_Relay/backend/IL_file/Air_Supply_Unit.il")
il = IL.read_text(encoding="utf-16").splitlines()

def show(a, b):
    print("\n".join(f"{i:4d}  {il[i - 1]}" for i in range(a, b + 1)))

show(1, 19)      # 에어백: P00154 에서 R00050 리셋, 엔코더로 증가
print("...")
show(104, 125)   # 노즐 전진 → 0.5초(Open_Dly 5) → 주입 SOL → (끝단 Rev_PX | 3초 | 비상 | T0109) 로 끝

### 7-2. 캐리어 에어 상태를 알려 줄 신호가 있나

`Device_Description.csv`에서 캐리어·에어·에어백 관련 I/O를 모두 뒤졌다. 캐리어 에어 상태를 직접 보는 것은 하나뿐이다.

| 주소 | 이름 | PLC 로직 | DB (10-17 하루) |
|---|---|---|---|
| **P0016F** | Air Bag 상승 CHK. L/S | `Conv_Limit Error`에서 쓰였다가 **2020-06-18 제거**(`AND _OFF`, "Airbag 리미트 에러 제거") | **항상 0** |
| P00130 | Air주입 Unit 캐리어 진입감지 | `LOAD P00130 AND _OFF` — 쓰지 않음. 대신 R00050 카운터로 트리거 | **항상 0** |
| P00243 | Air주입 Unit 에어주입1,2원통 SOL | 주입 SOL 출력 | 하루 약 4천 번 ON (사이클마다) |
| (없음) | 공압 압력 센서·압력 스위치 | — | 압력 스위치는 유압(`HYD*_PS`)·펌프 흡입뿐 |

같은 `Conv_Limit Error` 묶음의 `Mould_Clamp_Close_Fault_LS`, `Mould_Cylinder_Back_Fault_LS`도 함께 꺼졌다(입력 주소도 없음).
공정 순서로 보면 에어백은 **클로징 유닛이 뚜껑을 누르는 동안 부풀어(카운트 82~85, 클로징 상승은 89) 발포 중 금형을 잠그는 역할**로 보인다. P0016F가 그 뒤에서 "에어백이 올라와 있나"를 확인하던 스위치다.

In [ ]:
probe = db.fetch_day("2026-10-17", ["P00130", "P0016F", "P00243", "P00154", "R00050"], group="airbag_probe")
pd.DataFrame({c: {"0": int((probe[c] == 0).sum()), "1": int((probe[c] == 1).sum()), "켜짐 횟수": int((probe[c].diff() == 1).sum())}
              for c in ["P0016F", "P00130", "P00243", "P00154"]}).T

### 7-3. 에어백 유닛 사이클을 대차에 붙이기

에어백 유닛은 코어에 자기 진입 센서가 없다. PLC가 P00154 통과 뒤 카운트 82~85에서 스톱퍼를 내므로, **직전 P00154 통과의 대차**를 붙인다(`replay.attach_wagon`, 전진·후진과 같은 방식).
통과 → 스톱퍼 ON 지연이 한 덩어리(8.7~9.2초, 대차 간격 약 13초보다 짧음)인지 확인한다.

재생은 기간마다 캐시(`cache/replay/c01_units_*.pkl`)에 저장한다. 처음 한 번은 기간당 약 3분(DB 포함).

In [ ]:
RCACHE = paths.CACHE_DIR / "replay"; RCACHE.mkdir(parents=True, exist_ok=True)
CONV_TAGS = db.model_tags("conveyor", "v2.2")
PERIODS = [("2026-09-10", "2026-09-24"), ("2026-10-01", "2026-10-15"), ("2026-10-16", "2026-10-27")]

def units_period(a, b):
    f = RCACHE / f"c01_units_{a}_{b}.pkl"
    if f.exists():
        return pd.read_pickle(f)
    raw = db.fetch_range(a, pd.Timestamp(b) + pd.Timedelta(days=1), CONV_TAGS, group="conveyor")
    Pp, _, cc = replay.conveyor_core(raw, eval_every="100000min", progress=False, units=("md_fwd", "md_rev", "airbag"))
    hz = raw[["Hz_In_Conv"]].rename(columns={"Hz_In_Conv": "Hz"})
    out = {"passes": Pp, "cycles": cc.unit_cycles, "hz": hz}
    pd.to_pickle(out, f)
    return out

def airbag_table(per):
    a = replay.attach_wagon(per["cycles"], per["passes"], "airbag").dropna(subset=["wagon"])
    a = a[a.lag_s < 20].copy()
    h = per["hz"].reset_index().rename(columns={"Time": "t"}); h["t"] = h.t.astype("datetime64[ns]")
    a["t"] = a.t.astype("datetime64[ns]")
    a = pd.merge_asof(a.sort_values("t"), h.sort_values("t"), on="t", direction="backward")
    a["hzb"] = a.Hz.round(-1); a["wagon"] = a.wagon.astype(int)
    a["day"] = (a.t - pd.Timedelta(hours=6)).dt.normalize()
    return a

AB = pd.concat([airbag_table(units_period(a, b)) for a, b in PERIODS], ignore_index=True)
AB.groupby(AB.day.dt.month).lag_s.describe()[["count", "25%", "50%", "75%"]].round(2)

### 7-4. 컨베이어 속도가 대차마다 다르다

컨베이어 속도 지령(`Hz_In_Conv`)이 특정 대차가 지날 때 낮아진다(제품별 속도로 보임).
에어 주입 시간은 속도에 묶여 있으므로(모델 수정 예정 C-01) 대차를 비교하기 전에 속도를 빼야 한다.

In [ ]:
share = (AB[AB.day >= "2026-10-16"].groupby("wagon").hzb.value_counts(normalize=True).unstack().fillna(0) * 100).round(0)
share.loc[[1, 2, 5, 8, 9, 18, 23, 28, 29, 35]]

### 7-5. 속도를 뺀 대차별 주입 시간 — C01(대차 28)·C02(대차 18)

사이클마다 **주입 시간 − (그날 같은 속도 단계의 평균)** = 속도 보정 잔차. 대차·날짜별로 평균한다.
누수가 주입 시간을 늘린다면, 누수가 있던 날 그 대차의 잔차가 위로 튀어야 한다.

In [ ]:
def resid_table(df, col):
    d = df.dropna(subset=[col]).copy()
    grp = d.groupby(["day", "hzb"])[col]
    d["res"] = d[col] - grp.transform("mean")
    d = d[grp.transform("count") >= 200]                     # 사이클이 적은 속도 단계는 뺀다
    t = d.groupby(["day", "wagon"]).res.agg(["mean", "count"])
    return t["mean"].where(t["count"] >= 20).unstack("day")

R = resid_table(AB, "air_time")
R.columns = [c.strftime("%m-%d") for c in R.columns]
noise = R.sub(R.mean(axis=1), axis=0).stack().abs().quantile([0.5, 0.95])
print("대차 자기 평균 대비 하루 흔들림 |Δ| 중앙값·95%:", noise.round(3).to_dict())
R.loc[[18, 28], [c for c in R.columns if c >= "10-13"]].round(3)

In [ ]:
win = [c for c in R.columns if c >= "10-13"]
fig, ax = plt.subplots(figsize=(16, 9))
im = ax.imshow(R[win].values, aspect="auto", cmap="RdBu_r", vmin=-0.3, vmax=0.3)
ax.set_xticks(range(len(win)), win, rotation=90); ax.set_yticks(range(42), R.index)
for w, (d0, d1) in {28: ("10-17", "10-21"), 18: ("10-26", "10-26")}.items():
    y = list(R.index).index(w)
    ax.add_patch(plt.Rectangle((win.index(d0) - 0.5, y - 0.5), win.index(d1) - win.index(d0) + 1, 1, fill=False, lw=2, ec="k"))
plt.colorbar(im, ax=ax, label="속도 보정 잔차 (초)")
ax.set_title("에어백 유닛 주입 시간 — 대차 × 날짜 (검은 상자: 누수가 있던 날, 일지·데이터 기준)")
fig.tight_layout(); fig.savefig(FIGS / "C01_airbag_airtime_heatmap.png", dpi=110, bbox_inches="tight")

### 7-6. 2라인 캐리어 누수 기록 6건 전부

`전체` 시트의 2라인 캐리어 에어 누수·에어백·호스 기록이다. 누수가 있던 날(일지일, C01은 데이터상 10-17~10-21)과 같은 대차의 나머지 날을 비교한다.
에어백 유닛의 다른 지표(끌림·걸림 거리, 노즐 후진 시간)도 같이 본다.

In [ ]:
CASES = [  # (작업ID, 대차, 누수가 있던 날들, 조치)
    ("M-01-12-3", 35, ["09-10"], "CUSHG 하단 호스 교체 (19:40)"),
    ("C-062", 2, ["09-15"], "에어 누수 점검"),
    ("M-01-04-3", 10, ["09-22"], "CUSHG 에어백 라인 호스 절단·피팅 (19:40)"),
    ("C-122/124", 34, ["10-05"], "에어백 교체 (11:30)"),
    ("X-029 (C01)", 28, ["10-17", "10-19", "10-20", "10-21"], "리미트 스위치 점검 / 10-22 해소"),
    ("G-02-003 (C02)", 18, ["10-26"], "에어백 점검 (11:00)"),
]
metrics = ["air_time", "drag_cnt", "catch_cnt", "nozzle_rev_time_1", "nozzle_rev_time_2"]
tabs = {m: resid_table(AB, m) for m in metrics}
for m in tabs:
    tabs[m].columns = [c.strftime("%m-%d") for c in tabs[m].columns]
rows = []
for job, w, leak, act in CASES:
    row = {"작업ID": job, "대차": w, "조치": act}
    for m, T in tabs.items():
        s = T.loc[w].dropna()
        on, off = s[s.index.isin(leak)], s[~s.index.isin(leak)]
        row[f"{m} 누수일"] = on.mean()
        row[f"{m} 나머지(중앙값)"] = off.median()
        row[f"{m} 차이/흔들림"] = (on.mean() - off.median()) / (1.4826 * (off - off.median()).abs().median())
    rows.append(row)
cases = pd.DataFrame(rows).set_index("작업ID")
cases[[c for c in cases.columns if c.startswith("air_time") or "차이" in c]].round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharey=True)
for ax, (job, w, leak, act) in zip(axes.flat, CASES):
    s = tabs["air_time"].loc[w].dropna()
    d0 = list(s.index).index(leak[0]) if leak[0] in s.index else 0
    sel = s.iloc[max(0, d0 - 6): d0 + 7]
    ax.bar(range(len(sel)), sel.values, color=["tab:red" if c in leak else "tab:gray" for c in sel.index])
    ax.set_xticks(range(len(sel)), sel.index, rotation=90, fontsize=8)
    ax.axhline(0, color="k", lw=0.5); ax.set_title(f"대차 {w} — {job}\n{act}", fontsize=9)
axes[0, 0].set_ylabel("주입 시간 속도 보정 잔차 (초)"); axes[1, 0].set_ylabel("주입 시간 속도 보정 잔차 (초)")
fig.suptitle("에어백 유닛 주입 시간 — 누수가 있던 날(빨강) vs 같은 대차의 다른 날(회색)")
fig.tight_layout(); fig.savefig(FIGS / "C01_leak_cases_airtime.png", dpi=110, bbox_inches="tight")

### 7-7. 결론

1. **원리(PLC)**: 주입 시간은 "이동대가 대차에 끌려간 시간"(끝단 도착 또는 에어백 최대 3초)이다. 압력·유량 조건이 없다. 에어 누수로 길어질 이유가 없다.
2. **데이터**: 2라인 캐리어 누수 기록 6건(C01·C02 포함) 모두 누수일의 에어백 유닛 주입 시간(속도 보정)이 그 대차의 평소 흔들림 안이었다(차이 / 흔들림 −1.8 ~ +0.3, 5건은 오히려 짧음). 끌림·걸림 거리, 노즐 후진 시간도 일관된 변화가 없다.
3. **캐리어 에어 상태 신호**: P0016F(에어백 상승 L/S)·P00130은 항상 0이고 PLC 로직에서도 빠져 있다. 공압 압력 신호는 없다.
4. → **지금 수집되는 데이터로는 캐리어 에어 누수를 판별할 수 없다.** 필요한 것은 P0016F 복구(대차별 "에어백 안 올라옴")나 주입 라인 압력 센서다.
5. 부수: 컨베이어 속도가 대차에 묶여 있다(제품별 속도로 보임). 유닛 하루 평균 지표는 저속 대차가 몇 바퀴 돌았는지에도 흔들린다(모델 수정 예정 C-01).

정리본: [`cases/C01.md`](cases/C01.md) 5-1절.